# RAG Pipeline using LangChain — Fixed & Annotated

> **What was broken:** `youtube-transcript-api` v0.6+ removed the static `YouTubeTranscriptApi.get_transcript()` method. The class must now be **instantiated** and `fetch()` is used instead.

**Pipeline overview:**
1. **Indexing** — Fetch YouTube transcript → Split text → Embed → Store in FAISS
2. **Retrieval** — Similarity search over the vector store
3. **Augmentation** — Build a prompt with retrieved context
4. **Generation** — LLM answers from context only

## 0. Install Libraries

In [ ]:
# Install all required packages
# faiss-cpu: local vector store (no server needed)
# tiktoken: token counting for OpenAI models
%pip install -q \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-openai \
    faiss-cpu \
    tiktoken \
    python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 1. Configure API Keys

Set your OpenAI key. In Colab you can use `userdata`; locally use a `.env` file.

In [ ]:
import os

# ----- Option A: Colab Secrets (recommended for Colab) -----
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ----- Option B: .env file (recommended locally) -----
# from dotenv import load_dotenv
# load_dotenv()   # reads OPENAI_API_KEY from .env

# ----- Option C: paste directly (least secure, ok for quick tests) -----
os.environ["OPENAI_API_KEY"] ="sk-"   # <-- replace with your key

print("API key set:", bool(os.environ.get("OPENAI_API_KEY")))

API key set: True


## 2. Imports

In [ ]:
# youtube_transcript_api v0.6+ — must instantiate the class
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

print("All imports successful.")

All imports successful.


/tmp/ipykernel_5543/237919682.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a — Indexing: Document Ingestion (YouTube Transcript)

### ✅ Fix applied
| Old (broken) | New (correct) |
|---|---|
| `YouTubeTranscriptApi.get_transcript(video_id)` | `YouTubeTranscriptApi().fetch(video_id)` |

The class is now **instantiated** (`YouTubeTranscriptApi()`) before calling `fetch()`.  
Each element returned is a `FetchedTranscriptSnippet` object — access `.text` on each.

In [ ]:
# Only the video ID, not the full URL
# Example: https://www.youtube.com/watch?v=Gfr50f6ZBvo  ->  video_id = "Gfr50f6ZBvo"
video_id = "zduSFxRajkE"

try:
    # FIX: instantiate the class, then call .fetch()
    # In the original notebook this was incorrectly written as:
    #   YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    # That static method no longer exists in v0.6+
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # Each item is a FetchedTranscriptSnippet with a .text attribute
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(f"Transcript fetched. Total characters: {len(transcript)}")
    print("\nFirst 500 chars:\n", transcript[:500])

except TranscriptsDisabled:
    print("Transcripts are disabled for this video.")
except NoTranscriptFound:
    print("No English transcript found. Try another language code, e.g. languages=['en', 'en-US']")
except Exception as e:
    print(f"Unexpected error: {e}")

Transcript fetched. Total characters: 127302

First 500 chars:
 hi everyone so in this video I'd like us to cover the process of tokenization in large language models now you see here that I have a set face and that's because uh tokenization is my least favorite part of working with large language models but unfortunately it is necessary to understand in some detail because it it is fairly hairy gnarly and there's a lot of hidden foot guns to be aware of and a lot of oddness with large language models typically traces back to tokenization so what is tokeniza


## Step 1b — Indexing: Text Splitting

Split the transcript into overlapping chunks so each chunk fits within the LLM context window and related sentences stay together.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # max characters per chunk
    chunk_overlap=200   # overlap keeps context across chunk boundaries
)
chunks = splitter.create_documents([transcript])

print(f"Total chunks created: {len(chunks)}")
print("\nSample chunk (index 10):")
print(chunks[10].page_content)

Total chunks created: 159

Sample chunk (index 10):
here lowercase egg turns out to be a single token and in particular notice that the color is different so this is a different token so this is case sensitive and of course a capital egg would also be different tokens and again um this would be two tokens arbitrarily so so for the same concept egg depending on if it's in the beginning of a sentence at the end of a sentence lowercase uppercase or mixed all this will be uh basically very different tokens and different IDs and the language model has to learn from raw data from all the internet text that it's going to be training on that these are actually all the exact same concept and it has to sort of group them in the parameters of the neural network and understand just based on the data patterns that these are all very similar but maybe not almost exactly similar but but very very similar um after the EG demonstration here I have um an introduction from open a eyes chbt in Korean so m

## Step 1c & 1d — Indexing: Embed Chunks & Store in FAISS

`text-embedding-3-small` is fast and cheap. FAISS stores vectors locally in memory — no external DB needed.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

print("Vector store created.")
print(f"Number of indexed vectors: {len(vector_store.index_to_docstore_id)}")

Vector store created.
Number of indexed vectors: 159


In [ ]:
# Optional: peek at a stored document by its FAISS index ID
sample_id = list(vector_store.index_to_docstore_id.values())[0]
print(vector_store.get_by_ids([sample_id]))

[Document(id='e9d0bdb8-e623-4c21-969b-c5cc517df6e0', metadata={}, page_content="hi everyone so in this video I'd like us to cover the process of tokenization in large language models now you see here that I have a set face and that's because uh tokenization is my least favorite part of working with large language models but unfortunately it is necessary to understand in some detail because it it is fairly hairy gnarly and there's a lot of hidden foot guns to be aware of and a lot of oddness with large language models typically traces back to tokenization so what is tokenization now in my previous video Let's Build GPT from scratch uh we actually already did tokenization but we did a very naive simple version of tokenization so when you go to the Google colab for that video uh you see here that we loaded our training set and our training set was this uh Shakespeare uh data set now in the beginning the Shakespeare data set is just a large string in Python it's just text and so the questi

## Step 2 — Retrieval

The retriever searches FAISS for the `k` most similar chunks to a query using cosine similarity on embeddings.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}   # return top-4 most relevant chunks
)
print(retriever)

# Quick test
test_docs = retriever.invoke("What is DeepMind?")
print(f"\nRetrieved {len(test_docs)} chunks for test query.")
print("\nFirst chunk preview:\n", test_docs[0].page_content[:300])

tags=['FAISS', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a01e8089a60> search_kwargs={'k': 4}

Retrieved 4 chunks for test query.

First chunk preview:
 the best thing is to just wait for M bpe to becomes as efficient as possible and uh that's something that maybe I hope to work on and at some point maybe we can be training basically really what we want is we want tick token but training code and that is the ideal thing that currently does not exist


## Step 3 — Augmentation

Build the prompt. The LLM is instructed to answer **only from the transcript context** — this is what makes it a grounded RAG system rather than a free-form chat.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2   # low temperature = more factual, less creative
)

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, say you don't know.

{context}
Question: {question}
""",
    input_variables=["context", "question"]
)

In [ ]:
# Manual end-to-end test before building the full chain
question = "Is the topic is about tokenization"
retrieved_docs = retriever.invoke(question)

context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"context": context_text, "question": question})

answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic is about tokenization.


## Step 4 — Build the Full Chain (LCEL)

LangChain Expression Language (LCEL) chains everything together with the `|` pipe operator:

```
user_query
  → RunnableParallel  (retrieve context + pass through question)
  → PromptTemplate    (fill in context + question)
  → ChatOpenAI        (generate answer)
  → StrOutputParser   (extract plain string)
```

In [ ]:
def format_docs(retrieved_docs):
    """Join retrieved chunk texts into a single context string."""
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

# Step 1: run retriever & passthrough in parallel
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

# Step 2: assemble full pipeline
parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser

print("Chain assembled successfully.")

Chain assembled successfully.


In [ ]:
# Run the full chain
response = main_chain.invoke("Can you summarize the video?")
print(response)

The video discusses the complexities and challenges of tokenization in processing structured data, emphasizing the importance of understanding this stage due to potential security and AI safety issues. It highlights the use of different tokenization formats, such as the GPT-2 tokenizer, and introduces a web app for live tokenization demonstrations. The speaker mentions the distinction between hard and soft tokens and references a paper from OpenAI that explores visual tokenization. The video concludes with a hope for more efficient tokenization methods and the potential for future advancements in the field.


In [ ]:
# Interactive Q&A loop — ask any question about the video
print("RAG Q&A ready. Type 'quit' to exit.\n")
while True:
    q = input("Your question: ").strip()
    if q.lower() in ("quit", "exit", ""):
        break
    print("\nAnswer:", main_chain.invoke(q), "\n")

RAG Q&A ready. Type 'quit' to exit.

Your question: what is bpe

Answer: BPE stands for Byte Pair Encoding. It is a tokenization method that involves merging the most frequent pairs of bytes or characters in a sequence until no more merges are possible. This process helps in efficiently encoding and decoding text by creating a vocabulary of subword units. In the context provided, the BPE function is a key part of the encoding and decoding process, similar to the implementation discussed. 

Your question: tell me the most important concept from the video

Answer: The most important concept from the video is the significance of tokenization in processing structured data, particularly in the context of language models and AI. Tokenization involves breaking down data into manageable pieces (tokens), which can greatly affect efficiency, security, and the overall performance of models. Understanding the complexities and challenges of tokenization is crucial for working with AI systems. 

You

## Optional: Save & Reload the FAISS Index

So you don't have to re-embed every time you run the notebook.

In [33]:
# Save
vector_store.save_local("faiss_index")
print("Index saved to ./faiss_index/")

# Load later
# loaded_vs = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
# retriever = loaded_vs.as_retriever(search_type="similarity", search_kwargs={"k": 4})

Index saved to ./faiss_index/
